# Structured Streaming: Real-Time Data Processing

Welcome to the fifth notebook in our series! Up until now, we have focused on **Batch Processing**—where static datasets are processed all at once. Today, we enter the world of **Real-Time Data** using **Spark Structured Streaming**.

---

## 1. What is Spark Structured Streaming?

Structured Streaming is a scalable and fault-tolerant stream processing engine built on the Spark SQL engine. It allows you to express streaming computations the same way you express batch computations on static data.

### Key Concepts:
* **The Infinite Table:** Spark treats a live data stream as a table that is continuously being appended to. When new data arrives, it is as if a new row is inserted into an unbounded input table.
* **Micro-Batch Processing:** By default, Spark processes streaming data in small time-based batches (e.g., every 2 seconds), combining the throughput of batch processing with low-latency streaming.
* **Sources & Sinks:** 
  * *Sources* generate the stream (e.g., Apache Kafka, Amazon Kinesis, File directories, or socket connections).
  * *Sinks* write the output results (e.g., Console, Parquet files, Kafka, or Delta Lake tables).

## 2. Event-Time and Watermarking

When dealing with real-time data streams, network delays can cause data to arrive out of order.

* **Event Time:** The time when the event actually occurred on the device/source, rather than the time it was processed by the Spark cluster.
* **Watermarking:** A mechanism that tells Spark how long to wait for late-arriving data. Once the watermark passes a certain timestamp, Spark safely drops or finalizes aggregations for older windows, preventing memory from overflowing indefinitely.

## 3. Setting up the Environment

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp

# Initialize Spark Session with streaming capabilities
spark = SparkSession.builder \
    .appName("StructuredStreamingIntro") \
    .getOrCreate()

# Verify session
print("Spark Session initialized for streaming.")

## 4. Creating a Streaming Source (The Rate Source)

Spark provides a built-in test source called the **Rate Source** (`format("rate")`), which generates rows at a specified number of rows per second. This is ideal for testing streaming pipelines without needing an external message broker like Kafka.

In [ ]:
# Read stream from the built-in rate source (generates 2 rows per second)
streaming_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 2) \
    .load()

# Apply a standard narrow transformation (filter even values)
transformed_stream = streaming_df \
    .filter(col("value") % 2 == 0) \
    .select("timestamp", "value")

print("Streaming DataFrame schema:")
transformed_stream.printSchema()

## 5. Outputting to a Sink (The Console Sink)

To trigger the streaming execution loop, we use `.writeStream`. We will output our processed stream to the **console** so we can watch micro-batches print live as they arrive.

> **Note:** In an interactive notebook environment, running a continuous stream will block execution. You can stop the cell manually after observing a few micro-batches.

In [ ]:
# Write the stream to the console sink
query = transformed_stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .start()

# Await termination (in notebooks, use query.stop() or interrupt execution manually)
# query.awaitTermination(10) # Uncomment to run for 10 seconds automatically
# query.stop()

## Summary

In this notebook, we learned:
1. How Spark Structured Streaming treats live data streams as **unbounded tables**.
2. The mechanics of **micro-batch processing** for low-latency analytics.
3. How to use built-in test sources like the `rate` source and output streams using the `console` sink.